# Init Lakehouse

In [1]:
%%configure -f
{
    "defaultLakehouse": {"name": "DE_LH_100_BondedWarehouse"}
}

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, -1, Finished, Available, Finished)

# Init Imports (these need cutting-down post creation)

In [2]:
import os
import csv
import re
import shutil
import unicodedata
import pandas as pd

import notebookutils

#from decimal import Decimal
from datetime import datetime
from datetime import timedelta
#from collections import Counter
#from functools import reduce
import time

#from pyspark import StorageLevel
from pyspark.sql import DataFrame, Row
from pyspark.sql.functions import col, lit, when, concat, concat_ws, coalesce, count, monotonically_increasing_id, sum, to_date, udf, current_timestamp, length, substring, split, size, asc, row_number, desc, trim, regexp_replace
from pyspark.sql.functions import broadcast, hash, array, expr, array_distinct, date_format
from pyspark.sql.types import *
#from pyspark.sql import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 3, Finished, Available, Finished)

# Init Export Process

In [3]:
def save_dataframe_to_csv(df, file_path, show_header=False, mode='overwrite'):
    """
    Save a DataFrame as a single CSV file in a PySpark application.

    Parameters:
    df (pyspark.sql.DataFrame): The DataFrame to save.
    file_path (str): The path to save the CSV file.
    header (bool): Whether to include the header in the CSV file. Default is True.
    mode (str): The write mode. Options are 'overwrite', 'append', 'ignore', 'error' or 'errorifexists'. Default is 'overwrite'.

    Returns:
    None
    """

    use_pipes = len(df.columns) != 1
    print(f'Add pipes: {use_pipes}')
    print(f'Show headers: {show_header}')

    pandas_df = df.toPandas()
    
    # Replace newlines and carriage returns
    pandas_df = pandas_df.replace({r'\r\n': ' ', r'\n': ' ', r'\r': ' '}, regex=True)

    # Create a string representation of the DataFrame with '|' as separator
    # Escape special characters such as commas and pipes
    if use_pipes:
        csv_data = pandas_df.to_csv(sep="|", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")
    else:
        csv_data = pandas_df.to_csv(sep="~", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")

    # Add trailing pipe '|' at the end of each line
    csv_data_with_pipe = '\n'.join([line + '|' for line in csv_data.split('\n') if line])

    # Write to the file
    with open(file_path, 'w') as f:
        f.write(csv_data_with_pipe)


StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 4, Finished, Available, Finished)

# Init Debug & Incremental Vars

In [4]:
workspace_name = notebookutils.mssparkutils.env.getWorkspaceName()

if "DEV" in workspace_name.upper():
    debug = True
    incremental_run = False
    default_days_lag: int = 7

elif "UAT" in workspace_name.upper():
    debug = True
    incremental_run = True
    default_days_lag: int = 7

else:
    debug = False
    incremental_run = True
    default_days_lag: int = 0

if debug:
    print(debug , incremental_run)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 5, Finished, Available, Finished)

True False


# Init Days Lag Var

In [5]:
filterdate_pipe = ''

#default_days_lag: int = 1

enable_string_truncation = True
create_hash_cols: bool = False
transfer_file: bool = False
retain_error_records_in_ouput_file: bool = False

# Override Debug

#debug = False   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#debug = True   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

# Override Full Run 

#incremental_run = False    #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#incremental_run = True     #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 6, Finished, Available, Finished)

In [6]:
filterdate = datetime.now() - timedelta(days=default_days_lag)
filterdate = filterdate.date()

if debug:
    print(f'Get Products from: {filterdate}')

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 7, Finished, Available, Finished)

Get Products from: 2025-05-07


# Init Query(s)

In [7]:
products_df = spark.sql(f"""
SELECT 

'ENDCTG' AS Company_Code,
it.itemid AS Product_Code,
c.code AS Commodity_Code,

concat_ws('',
      substr(concat_ws('', pt.name, it.endfabriccomposition), 1, 119)
      ,
      CASE WHEN it.endmenswear = 1 and it.endwomenswear = 1 THEN 'U'  
            WHEN it.endmenswear = 1 THEN 'M' 
            WHEN it.endwomenswear = 1 THEN 'F' 
      ELSE '' END)
       as Description,

v.LangdonCode AS VAT_Rate_Identifier,
CAST(it.netweight AS Decimal(10, 4)) AS Unit_Weight,
'' AS Unit_Cost,
'' AS Unit_Cost_Currency,
'' AS Effective_Date,
'' AS Default_Import_Project,
'' AS Default_CLST_Project,
'' AS Default_IPR_Project,
CASE WHEN ci.additionalunits NOT IN (30,31) OR ci.additionalunits IS NULL THEN 30 ELSE ci.additionalunits END AS SKU,
'' AS BTI,
'' AS EC_Supplementary_Codes,
'' AS IP_Flag

FROM ecoresproduct p 

INNER JOIN inventtable it 
ON it.product = p.recid 

LEFT JOIN ecorescategory c 
ON c.recid = it.intrastatcommodity

LEFT JOIN ecoresproducttranslation pt 
ON pt.product = p.recid 

LEFT JOIN inventtablemodule itm 
ON it.itemid = itm.itemid 
AND it.dataareaid = itm.dataareaid
AND itm.moduletype = 1

LEFT JOIN vatgrouplookup v 
ON v.ItemVATgroup = itm.taxitemgroupid 

LEFT JOIN ecorescategoryintrastat ci 
ON ci.category = it.intrastatcommodity

WHERE it.dataareaid IN ('end.','END.')
AND GREATEST(
  p.modifieddatetime,
  c.modifieddatetime,
  it.modifieddatetime,
  itm.modifieddatetime,
  pt.modifieddatetime,
  ci.SinkModifiedOn
) >= '{filterdate}'

"""
)
if debug:
      display(products_df)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7f4e87b8-65b2-4615-be25-d19f276d9b8d)

In [8]:
products_df = products_df.select(
    substring(col("Company_Code").cast("string"),1, 10).alias("Company_Code"),
    substring(col("Product_Code").cast("string"),1, 25).alias("Product_Code"),
    substring(col("Commodity_Code").cast("string"),1, 10).alias("Commodity_Code"),
    substring(col("Description").cast("string"),1, 120).alias("Description"),
    substring(col("VAT_Rate_Identifier").cast("string"),1, 1).alias("VAT_Rate_Identifier"),
    substring(col("Unit_Weight").cast("string"),1, 11).alias("Unit_Weight"),
    col("Unit_Cost").cast("string").alias("Unit_Cost"),
    col("Unit_Cost_Currency").cast("string").alias("Unit_Cost_Currency"),
    col("Effective_Date").cast("string").alias("Effective_Date"),
    col("Default_Import_Project").cast("string").alias("Default_Import_Project"),
    col("Default_CLST_Project").cast("string").alias("Default_CLST_Project"),
    col("Default_IPR_Project").cast("string").alias("Default_IPR_Project"),
    substring(col("SKU").cast("string"),1,3).alias("SKU"),
    col("BTI").cast("string").alias("BTI"),
    col("EC_Supplementary_Codes").cast("string").alias("EC_Supplementary_Codes"),
    col("IP_Flag").cast("string").alias("IP_Flag")
)
if debug:
    display(products_df)

#display(products_df.select("Description").withColumn("Description_Length", length(col("Description"))).orderBy(desc("Description_Length")))

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 9c277138-1989-43fc-8ea3-e05540416902)

# N/A to NA

In [9]:
products_df = products_df.replace("N/A", "NA")

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 10, Finished, Available, Finished)

# Regex Pass to Remove Non-ASCII

In [10]:
products_df = products_df.withColumn("Description", trim(regexp_replace(regexp_replace("Description", "[^\\x00-\\x7F]|[\\|:#/?,Â%(),.-]", ""), "\\s+", " ")))

if debug:
    display(products_df)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 11, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 3f3e3b7d-d0dc-45c9-8a9d-6ef524a450f6)

## Date Field Changing Post Query(s)

In [11]:
list_date_columns_1 = [name for name, dtype in products_df.dtypes if dtype in ('date','timestamp')]

if debug:
    print("Date Columns to change: " , list_date_columns_1)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 12, Finished, Available, Finished)

Date Columns to change:  []


In [12]:
for column in list_date_columns_1:
    products_df = products_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 13, Finished, Available, Finished)

In [13]:
if debug:
    display(products_df)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 14, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7dbcf262-4e20-4469-990c-1a2edf79e5eb)

# Init Error Check Process

In [14]:
products_Mandatory_Columns = [
    "Company_Code",
    "Product_Code",
    "Commodity_Code",
    "VAT_Rate_Identifier",
    "SKU"
]

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 15, Finished, Available, Finished)

In [15]:
level = 1

null_condition = None
null_column_names_exprs = []

for column in products_Mandatory_Columns:
    condition = col(column).isNull()
    null_condition = condition if null_condition is None else null_condition | condition
    null_column_names_exprs.append(when(condition, lit(column)))

# Collect error columns into arrays
products_with_errors = products_df.withColumn("null_failed_columns", array(*null_column_names_exprs))

# Filter out nulls from those arrays
products_with_errors = products_with_errors.withColumn(
    "null_failed_columns", expr("filter(null_failed_columns, x -> x is not null)"))

# Generate the error messages (only when columns exist)
products_with_errors = products_with_errors.withColumn(
    "null_errors",
    when(size(col("null_failed_columns")) > 0,
         concat_ws("", lit("Columns "), concat_ws(" , ", col("null_failed_columns")), lit(f" are null at level {level}"))))

# Combine all errors
products_with_errors = products_with_errors.withColumn(
    "error_fields",
    concat_ws(" , ", col("null_errors"))
)

# Filter bad and good
products_bad_df = products_with_errors.filter(null_condition) \
    .drop("null_errors", "null_failed_columns")

products_df = products_with_errors.filter(~(null_condition)) \
    .drop("error_fields", "null_errors", "null_failed_columns")

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 16, Finished, Available, Finished)

In [16]:
if debug:
    display(products_bad_df)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 17, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 98ce68e1-7991-429f-a60d-1d7bcdac85bf)

In [17]:
products_bad_keys = [row["Product_Code"] for row in products_bad_df.select("Product_Code").distinct().collect()]

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 18, Finished, Available, Finished)

In [18]:
if debug:
    print("Level 1 Bad: " , products_bad_keys)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 19, Finished, Available, Finished)

Level 1 Bad:  ['AC45217', 'AC49371', 'AC44880', 'AC54043', 'AC36048', 'AC53046', 'AC54310', 'AC57063', 'AC54058', 'AC53340', 'AC53336', 'AC53355', 'AC33098', 'AC57059', 'AC52477', 'AC42509', 'AC54048', 'AC56831', 'AC52460', 'AC53352', 'AC33903', 'AC31871', 'AC47722', 'AC56842', 'AC54034', 'AC48641', 'AC44875', 'AC53331', 'AC56041', 'AC35098', 'AC53327', 'AC57066', 'AC53350', 'AC49386', 'AC52465', 'AC36868', 'AC44869', 'AC53656', 'AC25067', 'AC53338', 'AC34069', 'AC53361', 'AC52446', 'AC56835', 'AC37190', 'AC28343', 'AC34488', 'AC32086', 'AC53043', 'AC53024', 'AC28218', 'AC52434', 'AC57048', 'AC57529', 'AC37887', 'AC54055', 'AC54033', 'AC52464', 'AC44883', 'AC51966', 'AC53653', 'AC52391', 'AC47763', 'AC52399', 'AC26038', 'AC43429', 'AC43428', 'AC33319', 'AC27928', 'AC34103', 'AC51968', 'AC25607', 'AC34365', 'AC46522', 'AC48969', 'AC52392', 'AC48537', 'AC41655', 'AC53384', 'AC24268', 'AC56857', 'AC57303', 'AC56851', 'AC53363', 'AC54049', 'AC52421', 'AC34473', 'AC44873', 'AC52453', 'AC524

In [19]:
if debug:
    display(products_bad_df)
    display(products_df)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 20, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 0d1ffbdd-cbe8-4258-839e-b8f9bbdc0f5c)

SynapseWidget(Synapse.DataFrame, c96e775e-71c3-4668-82c5-0b413aa0ae1a)

# Init Good File Name & Date Logic

In [20]:
file_path_folder = "/lakehouse/default/Files/Output/"
file_extention = '.dat'

# file date time - stamp tomorrow's date if after 6.15pm --Nick: had to knock it back 1 hour to account for timezone difference; working
now = datetime.now()
cutoff_time = now.replace(hour=17, minute=15, second=0, microsecond=0)

if now > cutoff_time:
    tomorrow = now + timedelta(days=1)
    #file_datetime = tomorrow.strftime('%Y-%m-%d')
    file_datetime = tomorrow.strftime('%Y-%m-%d-%H')
else:
    #file_datetime = now.strftime('%Y-%m-%d')
    file_datetime = now.strftime('%Y-%m-%d-%H')


file_name = "products" + "_" + file_datetime + file_extention
file_path = file_path_folder + file_name

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 21, Finished, Available, Finished)

In [21]:
if debug:
    print("Now: " , now)
    print("Cutoff: " , cutoff_time)
    print("File Date: " , file_datetime)
    print("File Folder Path: " , file_path_folder)
    print("Good File Name: " , file_name)
    print("Good File Name: " , file_path)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 22, Finished, Available, Finished)

Now:  2025-05-14 11:08:22.696618
Cutoff:  2025-05-14 17:15:00
File Date:  2025-05-14-11
File Folder Path:  /lakehouse/default/Files/Output/
Good File Name:  products_2025-05-14-11.dat
Good File Name:  /lakehouse/default/Files/Output/products_2025-05-14-11.dat


# Remove Rows Already Sent

In [22]:
if incremental_run:
    
    # REMOVE ROWS FROM CURRENT RUN THAT HAVE ALREADY BEEN SENT (EXIST IN RECORD TRACKING)

    lakehouse_table_name = "bondedwarehouserecordtracking_products"
    container_column = "Product_Code"

    try:
        # Loads already sent Containers from record tracking
        sentrecords_df = spark.read.table(lakehouse_table_name).select(container_column).distinct()

        # Filter each input dataframe to EXCLUDE already sent
        products_df = products_df.join(sentrecords_df, products_df["Product_Code"] == sentrecords_df["Product_Code"], "left_anti")

    except Exception as e:
        print(f"An error occurred: {e}")

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 23, Finished, Available, Finished)

# Export Good

In [23]:
save_dataframe_to_csv(products_df, file_path, show_header=False)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 24, Finished, Available, Finished)

Add pipes: True
Show headers: False


# Init Bad File Name

In [24]:
error_file = "products_errors_" + file_datetime + file_extention
error_file_path = file_path_folder + error_file

if debug:
    print("file: ", error_file, " path: " , error_file_path)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 25, Finished, Available, Finished)

file:  products_errors_2025-05-14-11.dat  path:  /lakehouse/default/Files/Output/products_errors_2025-05-14-11.dat


# Export Bad

In [25]:
save_dataframe_to_csv(products_bad_df, error_file_path, show_header = True)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 26, Finished, Available, Finished)

Add pipes: True
Show headers: True


In [26]:
if products_df.take(1):

    ready_to_copy = True

else:

    ready_to_copy = False

if debug:
    print(ready_to_copy)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 27, Finished, Available, Finished)

True


# Init Record Tracking

In [27]:
if incremental_run:

    # Save the final_df to different tables based on the exportfile variable value

    def record_tracking_df_to_table(dataframe, table_name, file_name):
        """Saves distinct records to a table, checking for duplicates and enabling column mapping."""
        table_name_lower = table_name.lower()

        # Check if table exists
        table_exists = True
        try:
            spark.read.table(table_name_lower)
            print(f"Table {table_name_lower} exists.")
        except Exception as e:
            print(f"Table {table_name_lower} does not exist.")
            table_exists = False

        # Columns to deduplicate on (excluding metadata)
        dedup_cols = [col for col in dataframe.columns if col not in ["Timestamp", "ExportName", "ExportDate"]]

        if table_exists:
            try:
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))

                existing_df = spark.read.table(table_name_lower)

                distinct_existing_df = existing_df.dropDuplicates(subset=dedup_cols)
                initial_existing_count = distinct_existing_df.count()
                print(f"Existing distinct count: {initial_existing_count}")

                distinct_new_df = dataframe.dropDuplicates(subset=dedup_cols)

                combined_distinct_df = distinct_new_df.unionByName(distinct_existing_df) \
                                                    .dropDuplicates(subset=dedup_cols)
                final_distinct_count = combined_distinct_df.count()
                print(f"Final distinct count: {final_distinct_count}")

                rows_added = final_distinct_count - initial_existing_count
                print(f"Added {rows_added} new distinct records to {table_name_lower}.")

                combined_distinct_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

            except Exception as e:
                print(f"Exception: Saving all records in new table. Exception: {e}")
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

        else:
            try:
                print(f"Table doesn't exist. Saving all records in new table.")
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")
            except Exception as e:
                print(f"Error saving data: {e}")

    table_prefix = 'BondedWarehouseRecordTracking_'

    record_tracking_df_to_table(products_df, f"{table_prefix}products", file_name)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 28, Finished, Available, Finished)

# Init Send To Azure Blob Storage

In [28]:
if ready_to_copy == False:
    output_msg = f'Process Complete'

    notebookutils.notebook.exit(output_msg)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 29, Finished, Available, Finished)

## Copy the file to an ADLS account for loading to the SFTP
Set the source and destination paths

In [29]:
if "DEV" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_dev/ToBeSent/" + file_name
    
elif "UAT" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_uat/ToBeSent/" + file_name

else:
    dest_abfss_file_path = "Files/bonded_warehouse/ToBeSent/" + file_name

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 30, Finished, Available, Finished)

In [30]:
source_abfss_file_path = 'Files/Output/' + file_name

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 31, Finished, Available, Finished)

In [31]:
if transfer_file:
    notebookutils.fs.fastcp(source_abfss_file_path, dest_abfss_file_path)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 32, Finished, Available, Finished)

In [32]:
if ready_to_copy == True:
    output_msg = f'Process Complete'

notebookutils.notebook.exit(output_msg)

StatementMeta(, dfb75103-c7f0-4d6e-ab56-06fd3eefd90f, 33, Finished, Available, Finished)

ExitValue: Process Complete